# TP 2 — Premiers traitements distribués avec PySpark

**M1 · Deep Learning & Big Data · Séance 2 · Environnement : Databricks Free Edition**

Objectifs :
1. Lire une table de plusieurs millions de lignes stockée dans le catalogue Databricks.
2. Distinguer **transformations** (paresseuses) et **actions** (qui déclenchent le calcul).
3. Sélectionner, filtrer, créer des colonnes et agréger avec l'API DataFrame.
4. Retrouver les mêmes résultats en **Spark SQL**.
5. Voir **MapReduce** à l'œuvre sur un comptage de mots.
6. Enregistrer un résultat dans une table.

**Importer ce notebook :** dans Databricks, *Workspace › (votre dossier) › ⋮ › Import* puis choisissez ce fichier `.ipynb`. Attachez-le au calcul **Serverless**.

**Mode d'emploi :** complétez les `____` des cellules `# TODO` et répondez aux questions ❓.

> ℹ️ Dans Databricks, la variable `spark` (la SparkSession) et la fonction `display()` existent déjà.

---
## Partie 1 — Lire les données

Databricks fournit un catalogue d'exemples nommé `samples`. On utilise la table `samples.nyctaxi.trips` : des courses de taxi à New York.

Le nom suit le schéma **catalogue.schéma.table**.

In [ ]:
from pyspark.sql import functions as F

trajets = spark.read.table("samples.nyctaxi.trips")
trajets.printSchema()

In [ ]:
# Un aperçu : display() affiche un tableau interactif (tri, graphiques…)
display(trajets.limit(10))

In [ ]:
# Combien de lignes ? count() est une ACTION : elle déclenche un vrai calcul
nb_trajets = trajets.count()
print(f"{nb_trajets:,} trajets".replace(",", " "))

> 💡 Si les noms de colonnes affichés par `printSchema()` diffèrent de ceux utilisés dans la suite du TP, adaptez-les : le catalogue d'exemples peut évoluer.

Colonnes utilisées : `tpep_pickup_datetime` (prise en charge), `tpep_dropoff_datetime` (dépose), `trip_distance` (distance en miles), `fare_amount` (prix en dollars), `pickup_zip` et `dropoff_zip` (codes postaux).

---
## Partie 2 — Transformations et actions

Une **transformation** (`select`, `filter`, `withColumn`, `groupBy`…) renvoie un nouveau DataFrame **sans rien calculer** : Spark note seulement le plan.

Une **action** (`show`, `display`, `count`, `collect`, `write`…) déclenche le calcul de tout le plan.

In [ ]:
import time

debut = time.time()
longs = trajets.filter(F.col("trip_distance") > 10).select("trip_distance", "fare_amount")
print(f"Transformation : {time.time() - debut:.3f} s  ← rien n'a été calculé")

debut = time.time()
n = longs.count()
print(f"Action count() : {time.time() - debut:.3f} s  → {n} trajets de plus de 10 miles")

In [ ]:
# explain() montre le plan que Spark a préparé (à lire de bas en haut)
longs.explain()

### ❓ Question 2.1

Dans le plan affiché par `explain()`, repérez où apparaît le filtre sur `trip_distance`. Pourquoi est-il intéressant que Spark connaisse **tout** le plan avant de lancer le calcul ?

*Votre réponse :*

---
## Partie 3 — Manipuler un DataFrame

### 🛠️ Exercice 3.1 — Sélectionner et filtrer

Affichez la distance et le prix des courses **de plus de 20 dollars**, triées de la plus chère à la moins chère (10 premières).

In [ ]:
display(
    trajets
    .select(____, ____)                      # TODO : les deux colonnes distance et prix
    .filter(F.col("fare_amount") > ____)     # TODO : seuil de prix
    .orderBy(F.desc(____))                   # TODO : trier par prix décroissant
    .limit(10)
)

### 🛠️ Exercice 3.2 — Créer des colonnes

Ajoutez trois colonnes :
- `duree_min` : durée de la course en minutes ;
- `heure` : heure de prise en charge (0 à 23) ;
- `prix_par_mile` : prix divisé par la distance.

> 💡 `F.unix_timestamp(col)` convertit une date en nombre de secondes ; `F.hour(col)` extrait l'heure.
>
> ⚠️ Certaines courses ont une distance de 0 : une division classique provoquerait l'erreur `DIVIDE_BY_ZERO`. `F.try_divide(a, b)` renvoie une valeur nulle dans ce cas au lieu de planter.

In [ ]:
enrichi = (
    trajets
    .withColumn("duree_min",
                (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp(____)) / 60)   # TODO
    .withColumn("heure", F.hour(____))                                                      # TODO
    .withColumn("prix_par_mile", F.try_divide(F.col("fare_amount"), F.col(____)))           # TODO
)
display(enrichi.select("duree_min", "heure", "trip_distance", "fare_amount", "prix_par_mile").limit(10))

### 🛠️ Exercice 3.3 — Nettoyer avant d'analyser

Certaines courses ont une distance nulle ou une durée négative (erreurs de capteur). Gardez uniquement les courses dont la distance est **strictement positive** et la durée comprise **entre 1 et 180 minutes**. Combien de lignes avez-vous retirées ?

In [ ]:
propre = enrichi.filter(
    (F.col("trip_distance") > ____) &                 # TODO
    (F.col("duree_min").between(____, ____))          # TODO
)
retirees = nb_trajets - propre.count()
print("Lignes retirées :", retirees)

### 🛠️ Exercice 3.4 — Agréger

Pour chaque **heure** de la journée, calculez le nombre de courses, le prix moyen et la distance moyenne. Triez par heure.

Puis, dans le résultat de `display()`, cliquez sur **+ › Visualisation** pour tracer le nombre de courses en fonction de l'heure.

In [ ]:
par_heure = (
    propre
    .groupBy(____)                                            # TODO
    .agg(
        F.count("*").alias("nb_courses"),
        F.round(F.avg(____), 2).alias("prix_moyen"),          # TODO
        F.round(F.avg("trip_distance"), 2).alias("distance_moyenne"),
    )
    .orderBy("heure")
)
display(par_heure)

### ❓ Question 3.5

À quelles heures y a-t-il le plus de courses ? Le prix moyen varie-t-il selon l'heure ? Proposez une explication.

*Votre réponse :*

### 🛠️ Exercice 3.6 — Les trajets les plus fréquents

Trouvez les **10 couples (code postal de départ, code postal d'arrivée)** les plus fréquents.

In [ ]:
display(
    propre
    .groupBy(____, ____)            # TODO : les deux colonnes de codes postaux
    .count()
    .orderBy(F.desc("count"))
    .limit(10)
)

---
## Partie 4 — La même chose en SQL

Spark SQL et l'API DataFrame produisent **le même plan d'exécution**. On choisit selon ses préférences et son équipe.

In [ ]:
propre.createOrReplaceTempView("trajets_propres")

display(spark.sql('''
    SELECT heure,
           COUNT(*)                 AS nb_courses,
           ROUND(AVG(fare_amount), 2) AS prix_moyen
    FROM trajets_propres
    GROUP BY heure
    ORDER BY heure
'''))

### 🛠️ Exercice 4.1

Écrivez en SQL la requête de l'exercice 3.6 (les 10 trajets les plus fréquents).

In [ ]:
display(spark.sql('''
    SELECT ____, ____, COUNT(*) AS nb      -- TODO
    FROM trajets_propres
    GROUP BY ____                          -- TODO
    ORDER BY nb DESC
    LIMIT 10
'''))

---
## Partie 5 — MapReduce, version DataFrame

Le comptage de mots est le « Hello World » du Big Data. On le décompose en trois étapes :
1. **Map** : chaque ligne de texte est découpée en mots (une ligne par mot) ;
2. **Shuffle** : Spark regroupe sur une même machine toutes les occurrences d'un même mot ;
3. **Reduce** : on additionne les occurrences de chaque mot.

In [ ]:
phrases = spark.createDataFrame([
    ("le big data commence quand une machine ne suffit plus",),
    ("spark répartit les données et le calcul sur un cluster",),
    ("le deep learning a besoin de beaucoup de données",),
    ("une machine seule ne suffit plus pour le big data",),
], ["texte"])

# MAP : une ligne par mot
mots = phrases.select(F.explode(F.split("texte", " ")).alias("mot"))
display(mots)

In [ ]:
# SHUFFLE + REDUCE : regrouper par mot puis compter
comptage = mots.groupBy(____).count().orderBy(F.desc("count"))   # TODO
display(comptage)

### Où sont les morceaux ?

Un DataFrame est découpé en **partitions**, traitées en parallèle. La fonction `spark_partition_id()` indique dans quelle partition se trouve chaque ligne.

In [ ]:
display(
    trajets
    .withColumn("partition", F.spark_partition_id())
    .groupBy("partition").count()
    .orderBy("partition")
)

### ❓ Question 5.1

Pourquoi l'étape **shuffle** est-elle souvent la plus coûteuse dans un traitement distribué ?

*Votre réponse :*

---
## Partie 6 — Enregistrer un résultat

On enregistre l'agrégat par heure comme une **table** dans votre catalogue personnel (`workspace.default` dans Free Edition). Elle est stockée au format **Delta** (du Parquet enrichi d'un journal de transactions).

In [ ]:
par_heure.write.mode("overwrite").saveAsTable("workspace.default.tp2_courses_par_heure")

display(spark.read.table("workspace.default.tp2_courses_par_heure"))

Retrouvez la table dans le menu **Catalog › workspace › default** : vous pouvez l'interroger depuis l'éditeur SQL.

---
## 🎯 Pour vérifier que vous avez compris

1. Citez deux transformations et deux actions.
2. Pourquoi `filter(...)` est-il instantané alors que `count()` prend plusieurs secondes ?
3. Dans le comptage de mots, quelle ligne de code correspond au *map* ? au *reduce* ?

*Vos réponses :*